# 5 · Preference — *Mass-Mean-Probe* (PTO only)  `[TRAINING]`

Inside PTO's training signal: what, in words, the (chosen − rejected) preference pushes the policy
toward, and how that target moves across iterations and between K=0 and K=5. Per iteration the unit
**preference direction** = normalized mean(chosen − rejected) embedding; projecting words / MI-concepts
onto it reads out the preference. PTO-only by construction (GRPO has no pairs; its signal is in
`4_Training_and_Reliability`). Exports → `results/<VIEW>/figures|tables/5_preference/`.

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath("."))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 185, "display.max_columns", 50)
import eda_analysis
from eda_analysis import stats, behavior, training, pref, figures, plots

# ╔═══ VIEW — the one knob ════════════════════════════════════════════════════════╗
# "all" = every arm | "L0" = K=0 arms (PTO_LA0/GRPO_LA0) | "L5" = K=5 arms (thin, paused).
# Sets BOTH the arm filter AND the results root -> results/<VIEW>/figures|tables/<group>/.
# Edit the default for interactive use; render_views.py overrides it via the EDA_VIEW env var.
# (PTO-only notebook: GRPO has no preference pairs, so it analyses whichever PTO arms the view keeps.)
VIEW = os.environ.get("EDA_VIEW", "L0")

cfg = eda_analysis.EdaConfig(
    view=VIEW,                             # arm filter + results/<VIEW>/ root
    export_group="5_preference",           # topic family = this notebook's number
    selection="all",
    focus_arms=None, focus_metric="Q1Q2",
)
S = eda_analysis.notebook_setup(cfg)
FOCUS = cfg.focus_arms or sorted(S.SCORES.arm.unique())

## 0 · PTO arms with preference data

In [ ]:
PTO_ARMS = [a for a in S.ARMS if a.method == "PTO"
            and (cfg.focus_arms is None or a.label in cfg.focus_arms)
            and len(training.load_pref_pairs([a]))]
print("PTO arms analysed:", [a.label for a in PTO_ARMS])

## 1 · Per-arm preference, iteration by iteration  `[TRAINING]`
**Purpose.** Per PTO arm: probe quality (does the direction separate the pairs? wins > 0.5) → pooled
word ranking → per-iteration word drift (heatmap + top-words table) → **direction drift in 2D** +
consecutive cosine → **learned vs unlearned words** → MI-concept drift + a first→last read-out.

In [ ]:
RESULTS = {}   # arm -> {DIRS, CAT} for the K0-vs-K5 comparison in §2

def analyze_pref(arm):
    PAIRS = pref.add_text_features(training.load_pref_pairs([arm]))
    if PAIRS.empty:
        print(f"{arm.label}: no pairs."); return
    EMB = pref.embed_pairs(PAIRS)
    DIRS = pref.preference_direction_by_iter(EMB)
    print(f"\n################  {arm.label}  ################")
    PQ = pref.probe_quality_by_iter(EMB, DIRS)
    print("[probe] wins_correct should be > 0.5 for a real preference axis:"); display(PQ.round(4))
    eda_analysis.save_table(PQ.round(4), f"{arm.label}_pref_probe_quality", caption=f"{arm.label} preference-probe quality per iteration (wins_correct, gap, margin).")
    words, wmat = pref.embed_vocab(pref.build_vocab(PAIRS, top_n=3000)); WP = pref.word_projection(words, wmat, DIRS)
    # overall preference + per-iteration word drift
    fig = pref.pref_word_ranking(WP, title=f"{arm.label}: words by preference projection (green=chosen, red=rejected)")
    if fig: eda_analysis.save_fig(fig, f"{arm.label}_pref_word_ranking", caption=f"{arm.label} top chosen/rejected-aligned words (Mass Mean Probe, pooled over iters)."); plt.show()
    fig = pref.pref_word_drift_heatmap(WP, title=f"{arm.label}: preferred-word drift across iterations")
    if fig: eda_analysis.save_fig(fig, f"{arm.label}_pref_word_drift", caption=f"{arm.label} per-iteration projection of the top chosen-/rejected-aligned words (drift)."); plt.show()
    display(pref.top_words_by_iter(WP, k=8))
    # direction drift (vectors) + learned/unlearned words
    fig = pref.plot_direction_drift(pref.preference_direction_drift(DIRS), title=f"{arm.label}: preference-direction drift")
    if fig: eda_analysis.save_fig(fig, f"{arm.label}_pref_direction_drift", caption=f"{arm.label} preference direction in 2D PCA + consecutive cosine (how the preferred axis re-orients across iterations)."); plt.show()
    fig = pref.plot_learn_unlearn(pref.learn_unlearn_words(WP, k=8))
    if fig: eda_analysis.save_fig(fig, f"{arm.label}_pref_learn_unlearn", caption=f"{arm.label} words most newly preferred (learned) vs dropped (unlearned) across iteration transitions."); plt.show()
    # MI-concept drift + first->last read-out
    CAT = pref.category_projection(DIRS)
    if not CAT.empty:
        fig = pref.plot_category_drift(CAT)
        if fig: eda_analysis.save_fig(fig, f"{arm.label}_pref_category_drift", caption=f"{arm.label} MI-concept word groups projected onto the chosen-rejected direction per iteration."); plt.show()
        eda_analysis.save_table(CAT.round(4), f"{arm.label}_pref_MI_concepts", caption=f"{arm.label} MI-concept projection onto the preference direction per iteration.")
        wide = CAT.pivot_table(index="category", columns="train_iter", values="score")
        f0, fl = wide.columns.min(), wide.columns.max()
        print(f"[MI-concept shift iter {f0}->{fl}; + = MORE preferred over training]:")
        for cat, d in (wide[fl] - wide[f0]).sort_values(ascending=False).items():
            print(f"   {cat:<16} {wide.loc[cat, f0]:+.3f} -> {wide.loc[cat, fl]:+.3f}   (Δ {d:+.3f})")
    RESULTS[arm.label] = {"DIRS": DIRS, "CAT": CAT}

for arm in PTO_ARMS:
    analyze_pref(arm)
if not PTO_ARMS:
    print("No PTO arm with preference pairs scored yet.")

## 2 · Does look-ahead change *what* is preferred? K=0 vs K=5  `[TRAINING]`

> ⚠️ **DESCRIPTIVE only — PTO_LA5 is thin (4 iters).** This K=0-vs-K=5 overlay is **hypothesis-generating**, not an inferential K test; read a divergence as a direction to confirm once the K=5 arm finishes, not as a tested effect. (Same caveat as `4_Training` §4 / `6_Stats` §4.)

**Purpose.** Overlay PTO_LA0 vs PTO_LA5 MI-concept preference across iterations, and the cosine between
their per-iteration directions. **Read:** diverging curves / low cosine = look-ahead *suggests* steering the
preference toward different language.

In [ ]:
if {"PTO_LA0", "PTO_LA5"} <= set(RESULTS):
    cat0, cat5 = RESULTS["PTO_LA0"]["CAT"], RESULTS["PTO_LA5"]["CAT"]
    cats = sorted(set(cat0.category) & set(cat5.category))
    fig, axes = plt.subplots(1, len(cats), figsize=(2.7 * len(cats), 3.2), squeeze=False, sharey=True)
    for ax, c in zip(axes.flat, cats):
        for lab, cat, col in [("K=0", cat0, "#0072B2"), ("K=5", cat5, "#56B4E9")]:
            d = cat[cat.category == c]
            ax.plot(d.train_iter, d.score, marker="o", label=lab, color=col)
        ax.axhline(0, color="grey", lw=0.5, ls="--"); ax.set_title(c, fontsize=8); ax.set_xlabel("iter")
    axes.flat[0].legend(fontsize=7); axes.flat[0].set_ylabel("projection")
    fig.suptitle("MI-concept preference: K=0 vs K=5 (PTO)", y=1.05, fontweight="bold"); fig.tight_layout()
    eda_analysis.save_fig(fig, "pref_category_K0_vs_K5", caption="PTO MI-concept preference projection across iterations, K=0 vs K=5 — does look-ahead change what the policy prefers?"); plt.show()
    d0, d5 = RESULTS["PTO_LA0"]["DIRS"], RESULTS["PTO_LA5"]["DIRS"]
    common = sorted(set(d0) & set(d5))
    if common:
        print("cos(dir_K0, dir_K5) at matched iters:", {i: round(float(d0[i] @ d5[i]), 3) for i in common})
else:
    print("Need both PTO_LA0 and PTO_LA5 with pref pairs for the K comparison.")

## 3 · How to read this notebook
- **Probe real?** `wins_correct` ≫ 0.5 means the mean(chosen−rejected) direction genuinely separates the pairs.
- **What / how it drifts:** the word ranking + drift heatmap + learn/unlearn + MI-concept read-out test whether **affirmation/achievement** language becomes more preferred while **questions/reflection** fade — the latent-space signature of the behaviour drift in `3_Mechanism`.
- **Direction drift:** a low consecutive cosine means the preferred axis is re-orienting iteration to iteration, not just intensifying.
- **Caveat:** this is the **[TRAINING]** signal (what DPO optimises), not the eval — whether the shift is *good* is an eval/behaviour question (`1_Outcomes` / `3_Mechanism` / `6_Stats`).

In [ ]:
print("index ->", eda_analysis.build_index())